In [1]:
#IMPORTS
import pandas as pd
import numpy as np

# Modelling
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    make_scorer
)

# XGBoost
from xgboost import XGBClassifier

In [2]:
#LOAD DATA 
df = pd.read_csv("../data/alzheimers_disease_data.csv")
df.head()

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid


In [3]:
#DROP COLUMN
df = df.drop(columns=["PatientID", "DoctorInCharge"], errors="ignore")

# Check target distribution
print(df["Diagnosis"].value_counts(normalize=True))

df.info()

Diagnosis
0    0.646347
1    0.353653
Name: proportion, dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2149 entries, 0 to 2148
Data columns (total 33 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Age                        2149 non-null   int64  
 1   Gender                     2149 non-null   int64  
 2   Ethnicity                  2149 non-null   int64  
 3   EducationLevel             2149 non-null   int64  
 4   BMI                        2149 non-null   float64
 5   Smoking                    2149 non-null   int64  
 6   AlcoholConsumption         2149 non-null   float64
 7   PhysicalActivity           2149 non-null   float64
 8   DietQuality                2149 non-null   float64
 9   SleepQuality               2149 non-null   float64
 10  FamilyHistoryAlzheimers    2149 non-null   int64  
 11  CardiovascularDisease      2149 non-null   int64  
 12  Diabetes                   2149 n

In [4]:
#DEFINE FEATURE SETS
# Target
y = df["Diagnosis"]

# Feature groups (aligned with your thesis)
lifestyle_features = [
    "Age", "Gender", "Ethnicity", "EducationLevel", "BMI",
    "Smoking", "AlcoholConsumption", "PhysicalActivity",
    "DietQuality", "SleepQuality"
]

medical_features = [
    "FamilyHistoryAlzheimers", "CardiovascularDisease", "Diabetes",
    "Depression", "HeadInjury", "Hypertension", "SystolicBP",
    "DiastolicBP", "CholesterolTotal", "CholesterolLDL",
    "CholesterolHDL", "CholesterolTriglycerides"
]

cognitive_features = [
    "MemoryComplaints", "Confusion", "BehavioralProblems",
    "ADL", "Disorientation", "PersonalityChanges",
    "DifficultyCompletingTasks", "Forgetfulness"
]

# Create datasets
X_lifestyle = df[lifestyle_features]
X_lifestyle_med = df[lifestyle_features + medical_features]
X_full = df[lifestyle_features + medical_features + cognitive_features]

print(X_lifestyle.shape, X_lifestyle_med.shape, X_full.shape)

(2149, 10) (2149, 22) (2149, 30)


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

# Baseline model
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

# Feature sets dictionary
feature_sets = {
    "Lifestyle-only": X_lifestyle,
    "Lifestyle + Medical": X_lifestyle_med,
    "Full": X_full
}

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scoring metrics
scoring = {
    "roc_auc": "roc_auc",
    "precision": make_scorer(precision_score),
    "recall": make_scorer(recall_score),
    "f1": make_scorer(f1_score)
}

baseline_results = []

for name, X in feature_sets.items():
    scores = cross_validate(
        baseline_model,
        X,
        y,
        cv=cv,
        scoring=scoring
    )
    
    baseline_results.append({
        "Feature Set": name,
        "ROC-AUC Mean": scores["test_roc_auc"].mean(),
        "ROC-AUC SD": scores["test_roc_auc"].std(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean()
    })

baseline_df = pd.DataFrame(baseline_results)
baseline_df

,Feature Set,ROC-AUC Mean,ROC-AUC SD,Precision,Recall,F1
0,Lifestyle-only,0.519330,0.016742,0.368254,0.513158,0.428721
1,Lifestyle + Medical,0.530331,0.012427,0.383043,0.531579,0.445063
2,Full,0.796218,0.010390,0.561572,0.693421,0.620479


In [6]:
df["Diagnosis"].value_counts(normalize=True)

Diagnosis
0    0.646347
1    0.353653
Name: proportion, dtype: float64

In [ ]:
# Define all models
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        class_weight="balanced",
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=42
    )
}

results = []

for feature_set_name, X in feature_sets.items():
    for model_name, model in models.items():
        
        scores = cross_validate(
            model,
            X,
            y,
            cv=cv,
            scoring=scoring
        )
        
        results.append({
            "Feature Set": feature_set_name,
            "Model": model_name,
            "ROC-AUC Mean": scores["test_roc_auc"].mean(),
            "ROC-AUC SD": scores["test_roc_auc"].std(),
            "Precision": scores["test_precision"].mean(),
            "Recall": scores["test_recall"].mean(),
            "F1": scores["test_f1"].mean()
        })

results_df = pd.DataFrame(results)
results_df